<a href="https://colab.research.google.com/github/21centjoe/NELOS-Quantum-Vector/blob/main/yes%2C_make_the_collab_notebook_again_and_let_it_gi_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To achieve this level of dynamic variation, we need to completely decouple the engine from a single, hardcoded mathematical path.

This upgraded framework introduces an **audio-reactive parameter matrix**. When you change the musical selection, the script swaps out both the audio waveform type (Sine, Sawtooth, Square/Box) and the fractal trajectory coordinates. It handles the rendering across four distinct visual scenarios—including a highly unstable, symmetry-warping kaleidoscope mode—by dynamically altering the coordinate mapping equations in real time.

Paste and run this complete code block in your Colab cell:

In [1]:
from IPython.display import display, HTML

interface_html = """
<div style="background-color: #0d0d11; padding: 25px; border-radius: 12px; font-family: 'Courier New', monospace; text-align: center; color: #00ff66; max-width: 700px; margin: 0 auto; border: 1px solid #00ff66; box-shadow: 0 0 20px rgba(0,255,102,0.15);">
    <h2 style="margin-top: 0; text-transform: uppercase; letter-spacing: 2px;">Multi-Wave Fractal Control Console</h2>

    <div style="margin-bottom: 20px;">
        <label for="trackSelect" style="color: #aaa; font-size: 14px; margin-right: 10px;">SELECT TRANSMISSION CHANNEL:</label>
        <select id="trackSelect" style="background-color: #222; color: #00ff66; border: 1px solid #00ff66; padding: 8px; border-radius: 4px; font-family: monospace;">
            <option value="close_encounters">Close Encounters (Sine Wave / Deep Valley Zoom)</option>
            <option value="nice_guy_saw">No More Mr. Nice Guy (Sawtooth Wave / Sharp Needle Spiral)</option>
            <option value="nice_guy_box">No More Mr. Nice Guy (Square/Box Wave / Block Geometry)</option>
            <option value="kaleidoscope_crazy">CRITICAL MATRIX OVERLOAD (Variable / Kaleidoscope Mode)</option>
        </select>
    </div>

    <div style="margin-bottom: 20px;">
        <button id="startBtn" style="background-color: #00ff66; color: #000; border: none; padding: 12px 35px; font-size: 16px; font-weight: bold; border-radius: 4px; cursor: pointer; margin-right: 15px; font-family: monospace; box-shadow: 0 0 10px rgba(0,255,102,0.4);">INITIATE</button>
        <button id="stopBtn" style="background-color: #ff3333; color: #fff; border: none; padding: 12px 35px; font-size: 16px; font-weight: bold; border-radius: 4px; cursor: pointer; font-family: monospace; box-shadow: 0 0 10px rgba(255,51,51,0.4);">CEASE</button>
    </div>

    <div style="display: flex; flex-direction: column; align-items: center; gap: 15px;">
        <div>
            <canvas id="fractalCanvas" width="550" height="380" style="border: 1px solid #333; border-radius: 4px; background: #000;"></canvas>
        </div>
        <div>
            <canvas id="oscilloscopeCanvas" width="550" height="100" style="border: 1px solid #333; border-radius: 4px; background: #050505;"></canvas>
        </div>
    </div>
</div>

<script>
(function() {
    const fCanvas = document.getElementById('fractalCanvas');
    const fCtx = fCanvas.getContext('2d');
    const oCanvas = document.getElementById('oscilloscopeCanvas');
    const oCtx = oCanvas.getContext('2d');

    let isRunning = false;
    let animationFrameId = null;
    let audioSequenceTimeout = null;
    let zoomFactor = 1.0;
    let colorChannelIndex = 0;
    let lastColorSwapTime = 0;
    let stepCount = 0;

    let audioCtx = null;
    let analyserNode = null;

    // SCENARIO GEOMETRIES (Targets 180+ unique computational frames via dynamic paths)
    const scenarios = {
        close_encounters: { x: -0.743643887, y: 0.131825904, type: 'sine', speed: 1.03 },
        nice_guy_saw: { x: -0.122561167, y: 0.744861767, type: 'sawtooth', speed: 1.04 },
        nice_guy_box: { x: -0.75, y: 0.1, type: 'square', speed: 1.02 },
        kaleidoscope_crazy: { x: 0.0, y: 0.0, type: 'triangle', speed: 1.05 }
    };

    // MELODIC BAR PATTERNS
    const melodies = {
        close_encounters: {
            notes: [392.00, 440.00, 349.23, 174.61, 261.63],
            durations: [0.35, 0.35, 0.35, 0.35, 0.9],
            delays: [0.0, 0.4, 0.8, 1.2, 1.6],
            loopTime: 3200
        },
        nice_guy: {
            // "No More Mr. Nice Guy" primary chorus bars hook line
            notes: [293.66, 293.66, 329.63, 349.23, 349.23, 329.63, 293.66, 261.63, 293.66],
            durations: [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.4, 0.6],
            delays: [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.2],
            loopTime: 3500
        }
    };

    function initAudio() {
        audioCtx = new (window.AudioContext || window.webkitAudioContext)();
        analyserNode = audioCtx.createAnalyser();
        analyserNode.fftSize = 1024;
        analyserNode.connect(audioCtx.destination);
    }

    function playTone(freq, startTime, duration, waveType) {
        if (!audioCtx || audioCtx.state === 'suspended') return;

        let osc = audioCtx.createOscillator();
        let gain = audioCtx.createGain();

        osc.type = waveType;
        osc.frequency.setValueAtTime(freq, startTime);

        // Custom protective gain curves depending on harshness of wave shape
        let maxVolume = waveType === 'sine' ? 0.25 : waveType === 'square' ? 0.12 : 0.16;

        gain.gain.setValueAtTime(0, startTime);
        gain.gain.linearRampToValueAtTime(maxVolume, startTime + 0.03);
        gain.gain.exponentialRampToValueAtTime(0.001, startTime + duration);

        osc.connect(gain);
        gain.connect(analyserNode);

        osc.start(startTime);
        osc.stop(startTime + duration);
    }

    function sequenceEngine() {
        if (!isRunning) return;

        const currentMode = document.getElementById('trackSelect').value;
        const targetWave = scenarios[currentMode].type;
        let selectedMelody = melodies.close_encounters;

        if (currentMode.startsWith('nice_guy')) {
            selectedMelody = melodies.nice_guy;
        } else if (currentMode === 'kaleidoscope_crazy') {
            // Rapid chaotic modulation path selection
            selectedMelody = Math.random() > 0.5 ? melodies.close_encounters : melodies.nice_guy;
        }

        let now = audioCtx.currentTime;
        for (let i = 0; i < selectedMelody.notes.length; i++) {
            // If crazy mode is active, randomize oscillators on each step
            let nativeWave = currentMode === 'kaleidoscope_crazy' ?
                ['sine','sawtooth','square','triangle'][Math.floor(Math.random()*4)] : targetWave;

            playTone(selectedMelody.notes[i], now + selectedMelody.delays[i], selectedMelody.durations[i], nativeWave);
        }

        audioSequenceTimeout = setTimeout(sequenceEngine, selectedMelody.loopTime);
    }

    // GRAPH COMPUTATION RUNTIME
    function drawFrame(timestamp) {
        if (!isRunning) return;

        const modeKey = document.getElementById('trackSelect').value;
        const config = scenarios[modeKey];

        // Shift rendering profiles continuously
        if (!lastColorSwapTime) lastColorSwapTime = timestamp;
        if (timestamp - lastColorSwapTime > 600) {
            colorChannelIndex = (colorChannelIndex + 1) % 3;
            lastColorSwapTime = timestamp;
        }

        const width = fCanvas.width;
        const height = fCanvas.height;
        const imgData = fCtx.createImageData(width, height);
        const data = imgData.data;

        stepCount++;
        zoomFactor *= config.speed;
        if (zoomFactor > 80000) zoomFactor = 1.0;

        let xSpan = 3.2 / zoomFactor;
        let ySpan = (3.2 * (height / width)) / zoomFactor;

        let minX = config.x - xSpan / 2;
        let minY = config.y - ySpan / 2;

        const maxIterations = 60;

        for (let py = 0; py < height; py++) {
            let cy = minY + (py / height) * ySpan;
            for (let px = 0; px < width; px++) {
                let cx = minX + (px / width) * xSpan;

                // SPECIAL TRANSFORMATION KALEIDOSCOPE MODULE
                if (modeKey === 'kaleidoscope_crazy') {
                    // Normalize space coordinates into radial polar values to enforce mirror symmetry lines
                    let kx = px - width / 2;
                    let ky = py - height / 2;
                    let r = Math.sqrt(kx*kx + ky*ky);
                    let theta = Math.atan2(ky, kx);

                    // Slice coordinate grid cleanly into rotating geometric segments
                    let segments = 6.0;
                    theta = Math.abs( (theta % (Math.PI * 2 / segments)) - (Math.PI / segments) );

                    // Project altered vectors back onto processing targets
                    cx = (r * Math.cos(theta + stepCount*0.01)) / (width * 0.2) - 0.5;
                    cy = (r * Math.sin(theta + stepCount*0.01)) / (height * 0.2) + 0.5;
                }

                let zx = 0.0;
                let zy = 0.0;
                let i = 0;

                // Math Equation Engine Loop
                while (zx * zx + zy * zy < 4.0 && i < maxIterations) {
                    let xtemp = zx * zx - zy * zy + cx;
                    zy = 2.0 * zx * zy + cy;
                    zx = xtemp;
                    i++;
                }

                const idx = (py * width + px) * 4;
                let intensity = i === maxIterations ? 0 : Math.floor((i / maxIterations) * 255);

                // Dynamically route the pixel streams depending on active system channels
                if (modeKey === 'nice_guy_saw') {
                    // Sawtooth selection enforces sharp duotone splits
                    data[idx]     = colorChannelIndex === 0 ? intensity : intensity * 0.2;
                    data[idx + 1] = colorChannelIndex === 1 ? intensity : 0;
                    data[idx + 2] = colorChannelIndex === 2 ? intensity : intensity * 0.8;
                } else if (modeKey === 'nice_guy_box') {
                    // High-contrast blocking for box waves
                    let highContrast = intensity > 128 ? 255 : 0;
                    data[idx]     = colorChannelIndex === 2 ? highContrast : 0;
                    data[idx + 1] = colorChannelIndex === 0 ? highContrast : 0;
                    data[idx + 2] = colorChannelIndex === 1 ? highContrast : 0;
                } else if (modeKey === 'kaleidoscope_crazy') {
                    // Maximum unpredictable chroma distribution
                    data[idx]     = Math.sin(intensity * 0.05 + stepCount * 0.1) * 127 + 128;
                    data[idx + 1] = Math.cos(intensity * 0.03 + stepCount * 0.08) * 127 + 128;
                    data[idx + 2] = Math.tan(intensity * 0.01 + stepCount * 0.05) * 127 + 128;
                } else {
                    // Standard isolated solid color switching
                    data[idx]     = colorChannelIndex === 0 ? intensity : 0;
                    data[idx + 1] = colorChannelIndex === 1 ? intensity : 0;
                    data[idx + 2] = colorChannelIndex === 2 ? intensity : 0;
                }
                data[idx + 3] = 255;
            }
        }

        fCtx.putImageData(imgData, 0, 0);
        drawOscilloscope(modeKey);

        animationFrameId = requestAnimationFrame(drawFrame);
    }

    function drawOscilloscope(modeKey) {
        const width = oCanvas.width;
        const height = oCanvas.height;

        oCtx.fillStyle = '#050505';
        oCtx.fillRect(0, 0, width, height);

        oCtx.strokeStyle = '#112211';
        oCtx.beginPath();
        oCtx.moveTo(0, height / 2); oCtx.lineTo(width, height / 2);
        oCtx.stroke();

        if (!analyserNode) return;

        const bufferLength = analyserNode.frequencyBinCount;
        const dataArray = new Uint8Array(bufferLength);
        analyserNode.getByteTimeDomainData(dataArray);

        // Map trace colors according to selected aesthetic configurations
        let traceColor = '#00ff66';
        if (modeKey === 'nice_guy_saw') traceColor = '#ff00ff';
        if (modeKey === 'nice_guy_box') traceColor = '#ffff00';
        if (modeKey === 'kaleidoscope_crazy') traceColor = `hsl(${stepCount % 360}, 100%, 50%)`;

        oCtx.lineWidth = 2;
        oCtx.strokeStyle = traceColor;
        oCtx.beginPath();

        const sliceWidth = width / bufferLength;
        let x = 0;

        for (let i = 0; i < bufferLength; i++) {
            const v = dataArray[i] / 128.0;
            const y = (v * height) / 2;

            if (i === 0) {
                oCtx.moveTo(x, y);
            } else {
                oCtx.lineTo(x, y);
            }
            x += sliceWidth;
        }

        oCtx.lineTo(width, height / 2);
        oCtx.stroke();
    }

    // SYSTEM ROUTING TRIGGERS
    document.getElementById('startBtn').addEventListener('click', function() {
        if (isRunning) return;
        isRunning = true;
        zoomFactor = 1.0;

        if (!audioCtx) initAudio();
        if (audioCtx.state === 'suspended') audioCtx.resume();

        sequenceEngine();
        animationFrameId = requestAnimationFrame(drawFrame);
    });

    document.getElementById('stopBtn').addEventListener('click', function() {
        isRunning = false;
        if (animationFrameId) cancelAnimationFrame(animationFrameId);
        if (audioSequenceTimeout) clearTimeout(audioSequenceTimeout);

        setTimeout(() => {
            fCtx.fillStyle = '#000000';
            fCtx.fillRect(0, 0, fCanvas.width, fCanvas.height);
            oCtx.fillStyle = '#050505';
            oCtx.fillRect(0, 0, oCanvas.width, oCanvas.height);
        }, 50);
    });
})();
</script>
"""

display(HTML(interface_html))

### What Changes Between Scenarios:

1. **The Sine Wave Mode:** Features a smooth zoom tracking a deep valley coordinate set. It generates clean tone intervals that form the classic 5-note structural progression.
2. **The Sawtooth Mode:** Uses sharp, aggressive tone shapes matched with a tightly twisting needle spiral coordinate target, coloring the landscape with duotone magenta/blue hues.
3. **The Box/Square Wave Mode:** Pushes flat, hard-capped sound values into the oscilloscope. The visual rendering switches to a block-like coordinate threshold, causing the geometry to render as solid, high-contrast structural pixels.
4. **The Kaleidoscope Mode:** Snaps the Cartesian graph grid into a radial system divided into six mirror-symmetric segments. It bypasses regular channel constraints, allowing color waves to travel across the patterns based on a shifting phase angle, creating a chaotic display synchronized to randomized wave parameters.